### 1. Imports

In [36]:
import os
import requests
from typing import TypedDict
from langchain_groq import ChatGroq

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


from langgraph.graph import StateGraph, START, END

### 2. Configurations

In [37]:
GROQ_KEY_PATH = r"C:\Users\user\Desktop\devendra-ai\work\key-vault\groq\groq-api.key"
with open(GROQ_KEY_PATH) as key_file:
    api_key = key_file.read().strip()
os.environ["OPENAI_API_KEY"] = openai_api_key

In [3]:
CIS_URL = r"http://localhost:7860/api/v1/run/eef87fb7-0ffe-4987-9be8-2e17813b7eb0"

In [38]:
with open(r"C:\Users\user\Desktop\devendra-ai\work\key-vault\nvd\api.key") as f:
    nvd_api_key = f.read().strip()
NVD_API_KEY = nvd_api_key

In [60]:
MODEL = "openai/gpt-oss-120b"
llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0,api_key=api_key,)

### 3. Build the Tools

#### CIS Tool

In [6]:
import os

import requests

url = "http://localhost:7860/api/v2/workflows"
payload = {
    "flow_id": "eef87fb7-0ffe-4987-9be8-2e17813b7eb0",
    "input_value": "what are the windows password requirements for a strong password?"
}
headers = {
    "Content-Type": "application/json",
    "x-api-key": "sk-qNz-70oQVfAE8wvyRadRAVpwVDbfBtxcj_ivZNHx_Ow",
}

response = requests.post(url, json=payload, headers=headers)
response.raise_for_status()
result = response.json()

# The text reply is in output.text:
print(result["output"]["text"])

##Direct Answer
The Windows password requirements for a strong password include:
- A password must be at least six characters in length.
- It must contain characters from three of the following categories:
  - English uppercase characters (A through Z)
  - English lowercase characters (a through z)
  - Base 10 digits (0 through 9)
  - Non-alphabetic characters (such as !, $, #, %)
  - Any Unicode character that does not fall under the previous four categories.

Best practices suggest using a minimum of an 8-character password for accounts using multi-factor authentication (MFA) and a 14-character password for accounts not using MFA.

##CIS Sections
- 5.2 Use Unique Passwords
- 1.1.4 Ensure 'Minimum password length' is set to '14 or more character(s)'
- 18.9.26.5 Ensure 'Password Settings: Password Length' is set to 'Enabled: 15 or more'

##Explanation
The password complexity requirements are designed to enhance security by making passwords harder to guess or crack. The requirement for 

In [7]:
import requests
from langchain_core.tools import tool


@tool
def ask_cis(question: str) -> str:
    """Query the Flowise workflow knowledge base for CIS benchmarks and security guidance.

    Args:
        question: The security question or topic to search.
    """
    url = "http://localhost:7860/api/v2/workflows"
    headers = {
        "Content-Type": "application/json",
        "x-api-key": "sk-qNz-70oQVfAE8wvyRadRAVpwVDbfBtxcj_ivZNHx_Ow",
    }
    payload = {
        "flow_id": "eef87fb7-0ffe-4987-9be8-2e17813b7eb0",
        "input_value": question,
    }

    try:
        response = requests.post(url, json=payload, headers=headers, timeout=30)
        response.raise_for_status()
        result = response.json()

        # Safely extract output text
        return result.get("output", {}).get("text", "No output returned.")
    except requests.RequestException as e:
        return f"Error executing CIS workflow request: {e}"

In [8]:
ask_cis.invoke("Tell me about proper shutdown of windows servers")

"##Direct Answer\nProper shutdown of Windows servers should involve using the appropriate user rights to initiate a shutdown command, ensuring that only authorized users can perform this action to prevent denial-of-service (DoS) conditions.\n\n##CIS Sections\n- **2.2.21 Ensure 'Force shutdown from a remote system' is set to 'Administrators'**\n- **2.2.37 Ensure 'Shut down the system' is set to 'Administrators, Users'**\n\n##Explanation\nThe policy setting for 'Force shutdown from a remote system' recommends that only trusted administrators have the ability to shut down a Windows server remotely. This restriction helps prevent unauthorized shutdowns that could lead to service interruptions or DoS conditions. \n\nOn the other hand, the 'Shut down the system' setting allows both Administrators and authorized Users to shut down the server. This setting should be carefully managed to prevent guests or unauthorized users from executing shutdown commands, which could also result in DoS condit

#### NVD Tool

In [61]:
@tool
def lookup_cve(keyword:str)->str:
    """Query the NVD API for CVEs matching a keyword."""
    print("[TOOL]CVE Tool Activated")
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    headers = {"apiKey": NVD_API_KEY} if NVD_API_KEY else {}
    params = {"keywordSearch": keyword, "resultsPerPage": 3}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        data = r.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return "No CVEs found."
        lines = []
        for v in vulns:
            c = v["cve"]
            lines.append(f"{c['id']}: {c['descriptions'][0]['value'][:180]}")
        return "\n".join(lines)
    except Exception as e:
        return f"CVE lookup failed: {e}"

In [62]:
lookup_cve.invoke("SMB")

[TOOL]CVE Tool Activated


'CVE-1999-1387: Windows NT 4.0 SP2 allows remote attackers to cause a denial of service (crash), possibly via malformed inputs or packets, such as those generated by a Linux smbmount command that \nCVE-1999-0225: Windows NT 4.0 allows remote attackers to cause a denial of service via a malformed SMB logon request in which the actual data size does not match the specified size.\nCVE-1999-0495: A remote attacker can gain access to a file system using ..  (dot dot) when accessing SMB shares.'

In [63]:
import requests
from langchain_core.tools import tool

CIS_API_URL = "http://127.0.0.1:8000/ask"


@tool
def cis_security_tool(query: str) -> str:
    """
    Answer CIS Benchmark and Windows security compliance
    questions using the CIS FastAPI assistant.

    Args:
        query: The user's CIS security question.
    """
    try:
        response = requests.post(
            CIS_API_URL,
            json={"query": query},  # Fixed: use query, not question
            timeout=60,
        )

        response.raise_for_status()
        data = response.json()

        return data.get("answer", str(data))

    except requests.exceptions.ConnectionError:
        return "Cannot connect to the CIS FastAPI server."

    except requests.exceptions.Timeout:
        return "The CIS API request timed out."

    except requests.exceptions.HTTPError as exc:
        return f"CIS API error: {exc} - {response.text}"

    except requests.exceptions.RequestException as exc:
        return f"Request failed: {exc}"

In [65]:
cis_security_tool.invoke("Password")

'**1. Answer**  \nThe CIS guidance for password management focuses on a layered approach that balances security with usability. Key elements include:\n\n- **Length & Complexity** – Minimum 14\u202fcharacters for accounts without MFA, 8\u202fcharacters for accounts with MFA. Passwords must contain large letters, small letters, numbers, and special characters, or be a passphrase.  \n- **Uniqueness** – Each account, especially local admin/root/service accounts, must use a unique password.  \n- **Secure Storage** – Credentials must be stored with salted hashing; reversible encryption is disabled.  \n- **History & Expiration** – Remember at least 24 previous passwords; enforce a maximum age of 30\u202fdays or fewer.  \n- **Lockout & Grace Period** – Lockout after 5 or fewer failed attempts, lockout duration ≥\u202f15\u202fminutes, grace period ≤\u202f8\u202fhours.  \n- **MFA** – Where possible, require multi‑factor authentication; otherwise, apply stricter password rules.  \n- **Passphrases

#### IP Config Tool

In [17]:
import subprocess

@tool
def ipconfig_tool() -> str:
    """
    Returns the Windows network configuration using the ipconfig command.
    """
    try:
        result = subprocess.run(
            ["ipconfig"],
            capture_output=True,
            text=True,
            check=True,
            shell=True
        )

        return result.stdout

    except subprocess.CalledProcessError as e:
        return f"Error executing ipconfig:\n{e.stderr}"

In [18]:
print(ipconfig_tool.invoke({}))


Windows IP Configuration


Ethernet adapter Ethernet0:

   Connection-specific DNS Suffix  . : deepcloud.in
   Link-local IPv6 Address . . . . . : fe80::befc:3b5:9243:4327%5
   IPv4 Address. . . . . . . . . . . : 10.33.143.127
   Subnet Mask . . . . . . . . . . . : 255.255.255.0
   Default Gateway . . . . . . . . . : 10.33.143.1



### 4. Agent Layer

In [66]:
planner = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a planner. Decide weather RAG and CVE lookup is required"
)

In [67]:
retrieval_agent = create_agent(
    model=llm,
    tools=[cis_security_tool],
    system_prompt="Use the CIS policy tool to retrieve CIS benchmark guidance. Use the provided tools mandatorily"
)

In [68]:
threat_agent = create_agent(
    model=llm,
    tools=[lookup_cve],
    system_prompt="Use CVE lookup tool when threat intelligence is needed. Use the provided tools mandatorily"
)

In [70]:
validator_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Validate the evidence and produce a concise final answer"
)

In [71]:
response = planner.invoke({ 'messages': 'what is the password requriement for windows?'})

In [72]:
print(response['messages'][1].content)

**Windows password requirements – what you’ll see by default and how you can change them**

| Setting | Default value (Windows 10/11 & Windows Server) | How it’s enforced |
|---------|-----------------------------------------------|-------------------|
| **Minimum password length** | **8 characters** (can be set to 1‑14 in the UI, but most admins keep it ≥8) | Local Security Policy → “Minimum password length” |
| **Password complexity** | **Enabled** (requires a mix of character types) | Local Security Policy → “Password must meet complexity requirements” |
| **Complexity rule details** | • At least **three** of the following four categories:<br> - Upper‑case letters (A‑Z)<br> - Lower‑case letters (a‑z)<br> - Digits (0‑9)<br> - Non‑alphanumeric symbols (e.g., !, @, #, $) <br>• Cannot contain the user’s account name or parts of the full name that exceed two characters | Same setting as above |
| **Maximum password age** | **90 days** (after which the user is forced to change it) | Local

### 5. State

In [73]:
class CyberState(TypedDict):
    query: str
    plan: str
    rag: str
    cves: str
    final: str

### 6. Nodes

In [74]:
def planner_node(state: CyberState):
    r = planner.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"plan": str(r)}

In [75]:
def rag_node(state: CyberState):
    r = retrieval_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"rag": str(r)}

In [76]:
def cve_node(state: CyberState):
    r = threat_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"cves": str(r)}

In [77]:
def validator_node(state: CyberState):

    prompt = f"""
User Query:
{state["query"]}

Plan:
{state.get("plan", "")}

RAG:
{state.get("rag", "")}

CVEs:
{state.get("cves", "")}

SCORE:
validation score

Produce the final validated response.
Also, give a score from 0 to 10
"""
    r = validator_agent.invoke({
        "messages":[
            {"role":"user", "content":prompt}
        ]
    })
    return {"final": r}

### 7. Workflow

In [78]:
graph = StateGraph(CyberState)

graph.add_node("planner", planner_node)
graph.add_node("rag", rag_node)
graph.add_node("cve", cve_node)
graph.add_node("validator", validator_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "rag")
graph.add_edge("rag", "cve")
graph.add_edge("cve", "validator")
graph.add_edge("validator", END)

In [79]:
app = graph.compile()

### 8. Execute

In [80]:
result = app.invoke({
    "query": "How can I harden Windows SMB services against ransomware?"
})

[TOOL]CVE Tool Activated
[TOOL]CVE Tool Activated
[TOOL]CVE Tool Activated
[TOOL]CVE Tool Activated


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `openai/gpt-oss-120b` in organization `org_01m4mz0nr1ed58zpb2s3j7a78m` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Requested 15503, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [30]:
result

{'query': 'How can I harden Windows SMB services against ransomware?',
 'plan': "{'messages': [HumanMessage(content='How can I harden Windows SMB services against ransomware?', additional_kwargs={}, response_metadata={}, id='f175e5cb-5d64-41b6-9774-36d499f7332b'), AIMessage(content='RAG (Retrieval-Augmented Generation) and CVE (Common Vulnerabilities and Exposures) lookup would be useful here to provide the most up-to-date and specific security measures and known vulnerabilities related to Windows SMB services.\\n\\nI will proceed with RAG and CVE lookup to gather detailed and current information on hardening Windows SMB services against ransomware.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 37, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_toke

In [ ]:
print(result["final"]["messages"][-1].content)

In [ ]:
result = app.invoke({
    "query": "How can I secure windows SMB against latest attacks?"
})

In [33]:
result

{'query': 'How can I secure windows SMB against latest attacks?',
 'plan': "{'messages': [HumanMessage(content='How can I secure windows SMB against latest attacks?', additional_kwargs={}, response_metadata={}, id='972212a2-d50e-4d1d-9c29-977c208acdfa'), AIMessage(content='RAG (Retrieval-Augmented Generation) and CVE (Common Vulnerabilities and Exposures) lookup would be required to provide the most up-to-date and detailed information on securing Windows SMB against the latest attacks.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 36, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_bcb4333501', 'id': 'chatcmpl-EG1RIkjLjus

In [ ]:
print(result["final"]["messages"][-1].content)

In [58]:
from langchain.tools import tool
import requests

@tool
def lookup_cisa_kev(keyword: str) -> str:
    url = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    data = response.json()

    results = []

    for item in data["vulnerabilities"]:
        text = (
            item["vendorProject"] +
            item["product"] +
            item["shortDescription"]
        ).lower()

        if keyword.lower() in text:
            results.append(
                f"CVE: {item['cveID']}\n"
                f"Vendor: {item['vendorProject']}\n"
                f"Product: {item['product']}\n"
                f"Description: {item['shortDescription']}\n"
                f"Required Action: {item['requiredAction']}"
            )

    return "\n\n".join(results) if results else "No KEV entries found."

ValueError: Function must have a docstring if description not provided.

In [ ]:
Tool 4

In [ ]:
Tool 5

In [ ]:
tools = ["ask_cis", "lookup_cve", "lookup_cisa_kev", "lookup_attack", "lookup_epss"]
cyber_agent = create_agent(llm, tools=tools, system_prompt="write_your_prompt")

In [ ]:
cyber_agent.invoke(m1)